In [ ]:
import pandas as pd
from os import listdir as ls
from matplotlib import pyplot as plt

from emu_renewal.constants import OUTPUTS_PATH
from emu_renewal.inputs import get_google_mobility, get_world_shp
from emu_renewal.plotting import G_MOB_DOMAIN_CMAP

In [ ]:
job_path = OUTPUTS_PATH / "46090067"
countries = ls(job_path)

In [ ]:
def get_proc_mob_corr(job_path, iso3, mob_location):
    no_mob_path = job_path / iso3 / "no_mob"
    procs = pd.read_hdf(no_mob_path / "spaghetti.h5", key="spaghetti")["process"]
    mob = get_google_mobility(iso3)
    mob["proc"] = procs.median(axis=1)
    return mob.corr()["proc"][mob_location]

def plot_proc_mob_corr(corrs):
    plt.style.use("default")
    world = get_world_shp()
    fig, ax = plt.subplots(1, 1, figsize=[15.5, 6])
    world["corrs"] = world["ISO_A3"].map(corrs)
    corr_avail = world[world["corrs"].notna()]
    corr_unavail = world[world["corrs"].isna()]
    corr_avail.plot(ax=ax, column=corr_avail["corrs"], cmap="coolwarm", legend=True, vmin=-1.0, vmax=1.0)
    corr_unavail.plot(ax=ax, color="w", hatch="///", edgecolor="whitesmoke")
    ax.set_xticks([])
    ax.set_yticks([])
    world.boundary.plot(ax=ax, color="black", linewidth=0.2);

In [ ]:
location = "residential"
corrs = {}
for c in countries:
    try:
        corrs[c] = get_proc_mob_corr(job_path, c, location)
    except:
        continue
plot_proc_mob_corr(corrs)